# Supervised Dataset Construction

This notebook constructs leakage-aware regression and direction datasets from historical AAPL OHLCV data. No model is trained in this phase.

## Objective

Generate quantitative features, construct future-only targets, remove unusable warm-up and target-tail rows transparently, split chronologically, and fit preprocessing on training data only.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd()
if not (project_root / 'ml').exists():
    project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from ml.data.ingestion import MarketDataIngestionService
from ml.data.yahoo import YahooFinanceProvider
from ml.supervised import build_supervised_dataset

## Load Historical AAPL Data

In [ ]:
symbol = 'AAPL'
raw_path = project_root / 'data' / 'raw' / f'{symbol}.csv'
if raw_path.exists():
    ohlcv = pd.read_csv(raw_path, parse_dates=['date'])
else:
    service = MarketDataIngestionService(YahooFinanceProvider())
    ohlcv = service.ingest(symbol, '2020-01-01', '2026-01-01')
ohlcv.head()

## Build One-Day Regression Dataset

In [ ]:
regression_dataset = build_supervised_dataset(
    ohlcv,
    target_type='regression',
    horizon=1,
)
print(regression_dataset.feature_names)
print(regression_dataset.metadata)

## Build Direction Dataset

In [ ]:
direction_dataset = build_supervised_dataset(
    ohlcv,
    target_type='direction',
    horizon=1,
)
print(direction_dataset.metadata)

## Inspect Feature Warm-up and Target Tail

Feature warm-up rows arise from lagged and rolling calculations. Target-tail rows arise because the future close is unavailable. Neither is filled or backfilled.

In [ ]:
print('feature warm-up rows:', regression_dataset.metadata['feature_warmup_rows'])
print('target tail rows:', regression_dataset.metadata['target_tail_rows'])
print('feature count:', regression_dataset.metadata['feature_count'])

## Chronological Split Date Ranges

In [ ]:
for name in ('train', 'validation', 'test'):
    dates = getattr(regression_dataset, f'dates_{name}')
    print(name, dates.min(), 'to', dates.max(), 'rows:', len(dates))

## Train-Only Preprocessing

The StandardScaler is fitted by the pipeline only on X_train. Validation and test matrices are transformed with those already-fitted parameters; they are never used to fit the scaler.

In [ ]:
print('fitted training means:', regression_dataset.preprocessor.scaler.mean_)
print('train columns:', regression_dataset.feature_names)
assert all('future_return' not in name and 'direction' not in name for name in regression_dataset.feature_names)

## Leakage Discussion and Scope

Future shifts occur only in explicit target-construction code. Features use historical information available at or before their timestamp. The pipeline preserves chronological ordering and separates targets from X.

No prediction model has been trained yet. This phase does not include model fitting, predictions, signals, backtesting, or investment recommendations.